# Titanic - Machine Learning from Disaster
## 1. Importação das Bibliotecas
Importa as bibliotecas, pandas e sklearn, e carrega os .csv como data frames. 

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


## 2. Análise Rapida
Verificação dos valores.

In [2]:
print(train.head())
print(train.shape)
print(train.dtypes)
print(train.isnull().sum())
print(train.duplicated().sum())
print()

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
(8

## 3. Análise Gráfica
Os gráficos e insights apresentados foram gerados através do **Dash Analyzer**, uma aplicação desenvolvida em Flask, Pandas e Plotly para automação de análise de dados. 

### Destaques da Ferramenta:
* **Limpeza Automatizada:** Executa o pré-processamento inicial e a estruturação de ficheiros tabulares.
* **Geração Visual:** Cria dashboards dinâmicos combinados com cartões de insights textuais gerados por inteligência artificial.
* **Interatividade por IA:** Integra um chat contextual para consultas profundas sobre os dados do projeto.

## 4. Limpeza de Dados | Geração de Atributos
Limpa valores nulls, utilizando mediana e moda, e limpeza total pra algumas colunas, Gera as colunas 'Sex' e 'Embarked' como números.

In [ ]:
train['Has_Cabin'] = train['Cabin'].notnull().astype(int)
train = train.drop(columns=['Cabin'])

test['Has_Cabin'] = test['Cabin'].notnull().astype(int)
test = test.drop(columns=['Cabin'])

porto_modal = train['Embarked'].mode()[0]
train['Embarked'] = train['Embarked'].fillna(porto_modal)
test['Embarked'] = test['Embarked'].fillna(porto_modal)

train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
for dataset in [train, test]:
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')

mediana_idade = train['Age'].median()
train['Age'] = train['Age'].fillna(mediana_idade)
test['Age'] = test['Age'].fillna(mediana_idade)

train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

if test['Fare'].isnull().sum() > 0:
    test['Fare'] = test['Fare'].fillna(train['Fare'].median())
    
train = pd.get_dummies(train, columns=['Sex', 'Embarked', 'Title'], drop_first=True)
test = pd.get_dummies(test, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

<>:11: SyntaxWarning: invalid escape sequence '\.'
<>:12: SyntaxWarning: invalid escape sequence '\.'
<>:11: SyntaxWarning: invalid escape sequence '\.'
<>:12: SyntaxWarning: invalid escape sequence '\.'
C:\Users\Guilherme Araujo\AppData\Local\Temp\ipykernel_16244\2204278132.py:11: SyntaxWarning: invalid escape sequence '\.'
  train['Title'] = train['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
C:\Users\Guilherme Araujo\AppData\Local\Temp\ipykernel_16244\2204278132.py:12: SyntaxWarning: invalid escape sequence '\.'
  test['Title'] = test['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


## 5. Treinamento dos Modelos e Submissão Final
Treinamento de modelos, avaliação e exportação do arquivo.

In [4]:
features = [
    'Pclass', 'Age', 'Fare', 'Has_Cabin',
    'Sex_male', 'Embarked_Q', 'Embarked_S',
    'FamilySize', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare'
]

X = train[features]
y = train['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_val_scaled)

print("--- Regressão Logística ---")
print(f"Acurácia: {accuracy_score(y_val, lr_preds):.4f}")
print(classification_report(y_val, lr_preds))

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_val)

print("--- Random Forest ---")
print(f"Acurácia: {accuracy_score(y_val, rf_preds):.4f}")
print(classification_report(y_val, rf_preds))

X_test = test.reindex(columns=features, fill_value=0)
final_predictions = rf_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': final_predictions
})

submission.to_csv('submission.csv', index=False)

--- Regressão Logística ---
Acurácia: 0.8324
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       105
           1       0.80      0.80      0.80        74

    accuracy                           0.83       179
   macro avg       0.83      0.83      0.83       179
weighted avg       0.83      0.83      0.83       179

--- Random Forest ---
Acurácia: 0.8380
              precision    recall  f1-score   support

           0       0.86      0.87      0.86       105
           1       0.81      0.80      0.80        74

    accuracy                           0.84       179
   macro avg       0.83      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179



### Projeto feito com base no desafio "Titanic - Machine Learning from Disaster" do kaggle.